In [31]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection/dataset
!ls

Cloning into 'Continual-hate-speech-detection'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 83 (delta 27), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 173.94 KiB | 4.70 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/Continual-hate-speech-detection/dataset
df_loader.py  stream_generator.py


In [34]:
import torch

from dataset.df_loader import (
    getdf_davidson,
    getdf_hatexplain
)

from dataset.stream_generator import (
    create_stream,
    merge_streams,
    online_stream,
    stream_summary,
    dataset_distribution
)

from transformers import (
    AutoTokenizer,
    AutoConfig
)

from models.model_builder import CustomClassifier

ModuleNotFoundError: No module named 'dataset.stream_generator'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

In [ ]:
print("Loading datasets...")

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print("Davidson:", df_dv.shape)
print("HateXplain:", df_hx.shape)

In [ ]:
dv_stream = create_stream(
    df=df_dv,
    batch_size=32,
    shuffle=True
)

hx_stream = create_stream(
    df=df_hx,
    batch_size=32,
    shuffle=True
)

In [ ]:
stream_summary(dv_stream, "Davidson Stream")
stream_summary(hx_stream, "HateXplain Stream")

In [ ]:
full_stream = merge_streams(
    streams=[dv_stream, hx_stream],
    shuffle_streams=False
)

stream_summary(full_stream, "FULL STREAM")
dataset_distribution(full_stream)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [ ]:
num_labels = 2

config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

In [ ]:
model = CustomClassifier(
    model_name="roberta-base",
    config=config,
    class_weights=None,
    use_lora=True
).to(device)

print("Model loaded")

In [ ]:
print("Starting stream simulation...\n")

stream_iterator = online_stream(full_stream)

for step, batch in enumerate(stream_iterator):

    print("=" * 50)
    print(f"STREAM STEP {step}")
    print("=" * 50)

    print(batch.head())

    print("\nBatch size:", len(batch))

    print("\nLabel distribution:")
    print(batch["label"].value_counts())

    # ---------------------------------
    # FUTURE:
    # train_step(batch)
    # ---------------------------------

    if step == 2:
        break